In [ ]:
# autoclasses - 2nd way to use hf models
# less abstract as compared to pipeline func that we covered last time

In [ ]:
! pip install datasets evaluate transformers diffusers accelerate ftfy pyarrow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


## Tokenizer

In [ ]:
# before sending input to the  model, we need to tokenize the input
# the tokenizer of that model takes care of tokenizing!

In [1]:
# in order to perform tokenization in hf, we have the `AutoTokenizer` class

from transformers import AutoTokenizer # class


checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint) # i am loading the tokenizer along with this model

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [2]:
# send both of these inputs "together" to the model

raw_inputs = [
    "I am learning Operating Systems and it's so interesting.", # more token ids let's say 14 token ids
    "I write terrible C++ code!",  # lesser token ids 11 token ids
] # text inputs that the model cannot ingest directly

inputs = tokenizer(raw_inputs, padding=True, truncation=True, max_length=15, return_tensors="pt")
print(inputs)



{'input_ids': tensor([[ 101, 1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 5875,
         1012,  102],
        [ 101, 1045, 4339, 6659, 1039, 1009, 1009, 3642,  999,  102,    0,    0,
            0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


In [ ]:
[3, 6, 4] # tensor corresp to the 1st sent
[7, 5] # tensor corresp to the 2nd sent

[
    [3, 6, 4],
    [7, 5, 0]
]


[[3, 6, 4], [7, 5, 0]]

In [3]:
# another way to do it - a closer look into the tokenizer
sequence = "I am learning Operating Systems and it's so funtastic."
tokens = tokenizer.tokenize(sequence)

print("--------TOKENIZED SENTENCE--------")
print(tokens)

print("\n\n--------TOKENS MAPPED TO THEIR IDS--------")
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

--------TOKENIZED SENTENCE--------
['i', 'am', 'learning', 'operating', 'systems', 'and', 'it', "'", 's', 'so', 'fun', '##tas', '##tic', '.']


--------TOKENS MAPPED TO THEIR IDS--------
[1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569, 10230, 4588, 1012]


padding=True, attention_mask

point 1: models work with matrices and vectors - mathmul!!!

point 2: 1st sent is longer than the 2nd sent -> 1st sent is going to have more token_ids as compared to 2nd

inp 1: [4, 7, 8] # inp 1 -> row 1
inp 2: [4, 9] # inp 2 -> row 2

[[4, 7, 8],
 [4, 9]] # not a matrix


[[4, 7, 8],
 [4, 9, 0]] # this is a matrix

impossible for one to construct a matrix
something that helps us construct a matrix out of unequal inputs

**TAKE A PAUSE!**   Time to Know About

**SPECIAL TOKENS!**

In [5]:
# already tokenized input

# mapping the token ids back to the tokens
decoded_string = tokenizer.decode([1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569, 10230, 4588, 1012])
print(decoded_string)

i am learning operating systems and it ' s so funtastic.


In [6]:
# raw text
decoded_string = tokenizer.decode([101, 1045, 2572, 4083, 4082, 3001, 1998, 2009, 1005, 1055, 2061, 4569,
         1012,  102])
print(decoded_string)

[CLS] i am learning operating systems and it ' s so fun. [SEP]


In [ ]:
Are these IDs generated by HF will be same as other LLMs?

## Models

In [7]:
from transformers import AutoModel, AutoTokenizer
import torch


checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english" # tokenizer one

# wrong: checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# vocab mismatch

model = AutoModel.from_pretrained(checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
outputs = model(**inputs) # inputs is a dictionary
# alternative syntax
# outputs = model(
#     input_ids=inputs["input_ids"],
#     attention_mask=inputs["attention_mask"]
# )
print(outputs.last_hidden_state.shape) # bs, seq_len, dim

# say a model has 20 layers
# new info: logits are calculated using the output of layer #20
# last_hidden_state gives us the output of layer #20 (which is NOT logits yet)

# text gen models are decoder only models
# input with 5 tokens ids -> model (5 embeddings) -> output of the last layer (5 hidden states - 5 vectors) ----- (something)----- -> a logits vector of size V -> probs -> logprobs -> token prediction
# output of 1st layer?? 5 emb
# output of 2nd layer?? 5 emb
# output of 3rd layer?? 5 emb


torch.Size([2, 14, 768])


In [ ]:
# 2 tokens

# 1 1st layer - emb layer - gives me the embeddings corresponding to these 2 tokens

# [
#     [-0.9, 1.0, 8] # emb of the 1st token
#     [3.0, 4.0, 5.0] # emb of the second token
# ] 2 (tokens) x 3 (dim of embedding)

In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english" # this is a model for sentiment analysis
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# Define tokenizer and inputs within this cell
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_inputs = [
    "I am learning Operating Systems and it's so interesting.",
    "I write terrible C++ code!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, max_length=15, return_tensors="pt")

outputs = model(**inputs) # model doesn't expect a dictionary, it expects input_ids and attn_mask

print("--------SHAPE OF LOGIT TENSOR--------")
print(outputs.logits.shape)

print("\n\n--------LOGIT TENSOR--------")
print(outputs.logits)

# convert logits to probs --> softmax (math func)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1) # probs
predictions = torch.argmax(predictions, dim=1)
print("\n\n--------PREDICTIONS--------")
print(predictions) # os one, c++ one

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--------SHAPE OF LOGIT TENSOR--------
torch.Size([2, 2])


--------LOGIT TENSOR--------
tensor([[-4.2611,  4.6028],
        [ 4.6717, -3.7884]], grad_fn=<AddmmBackward0>)


--------PREDICTIONS--------
tensor([1, 0])


In [11]:
# what is 1? negative or positive?
# figure out the mapping b/w numbers and their corresponding labels
# ans is in the api docs
# https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
print(model.config.id2label[1])
print(model.config.id2label[0])


POSITIVE
NEGATIVE
